## prepare


In [13]:
import numpy as np
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.utils.data import TensorDataset
import datasets
from datasets import load_from_disk, Image
import torch
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torchmetrics.image.fid import FrechetInceptionDistance
import math
import copy

In [14]:
import os
import random
import numpy as np
import torch


def set_seed(seed: int):
    # 1. Python random
    random.seed(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果你用多个 GPU

    # 4. cuDNN 设置为确定性模式，关闭 benchmark
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True





def seed_worker(worker_id):
    # 每个 worker 基于相同或偏移后的种子
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


# 在脚本最上方调用一次
SEED = 42
set_seed(SEED)

# PyTorch 1.7+ 可直接用 generator
gen = torch.Generator()
gen.manual_seed(SEED)

### dataset

In [15]:
cuda_num = 2
s = 28
p = 1
INPUT_DIM = p * s * s
LATENT_DIM = 5

HIDDEN_DIM = 500
data_folder = "data_mnist"
datasetname = "mnist"

# train+label
train_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/train")
print(f"Total samples: {len(train_dataset)}")
train_images = (
    torch.stack([torch.tensor(np.array(img)) for img in train_dataset["image"]]).float()
    / 255.0
)
train_images = train_images.reshape(-1, 28 * 28) # Flatten to [60000, 784]
train_labels = torch.tensor(train_dataset["label"], dtype=torch.long)
train_dataset_torch = TensorDataset(train_images, train_labels)



# test+label
test_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/test")
test_images = (
    torch.stack([torch.tensor(np.array(img)) for img in test_dataset["image"]]).float()
    / 255.0
)
test_images = test_images.reshape(-1, 28 * 28) # Flatten to [10000, 784]
test_labels = torch.tensor(test_dataset["label"], dtype=torch.long)
test_dataset_torch = TensorDataset(test_images, test_labels)


Total samples: 60000


In [16]:
from torch.utils.data import random_split

# 定义划分比例
train_size = int(0.9 * len(train_dataset_torch))  # 90%
val_size = len(train_dataset_torch) - train_size  # 10%

# 按比例划分数据集
train_dataset_torch, val_dataset_torch = random_split(train_dataset_torch, [train_size, val_size], generator=gen)

# 打印划分后的数据集大小
print(f"Train dataset size: {len(train_dataset_torch)}")
print(f"Validation dataset size: {len(val_dataset_torch)}")

Train dataset size: 54000
Validation dataset size: 6000


### vae class


In [17]:
class VAE_Encoder(nn.Module):
    """
    VAE的encoder，将输入的图片映射到隐变量的分布，输出隐变量分布的均值和方差.
    由于方差不能为负，所以这里输出的是log(方差)，而不是方差本身.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Encoder, self).__init__()

        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.Tanh())

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)


class VAE_Decoder(nn.Module):
    """
    VAE的decoder，将隐变量映射回图片.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Decoder, self).__init__()

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.decoder(z)


class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super(VAE, self).__init__()

        self.latent_dim = latent_dim

        self.encoder = VAE_Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = VAE_Decoder(input_dim, hidden_dim, latent_dim)

    def encode(self, x):
        """
        将输入的图片映射到隐变量的分布，输出隐变量分布的均值和log(方差).
        """
        return self.encoder(x)

    def decode(self, z):
        """
        将隐变量映射回图片.
        """
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        """
        重参数化技巧，先从标准正态分布中采样一个epsilon，然后根据隐变量分布的均值和方差，计算出隐变量.
        """
        std = torch.exp(
            0.5 * logvar
        )  # 计算标准差, std = sqrt(var) = sqrt(exp(logvar)) = exp(logvar/2)
        epsilon = torch.randn_like(
            std, requires_grad=False
        )  # 从标准正态分布中采样epsilon
        return mu + epsilon * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

### loss


In [18]:
def loss_iwae_gauss(mu, logvar, x, decoder, k=5, sigma2=1.0, device=None):
    if device is None:
        device = x.device

    B = x.size(0)
    D = x.view(B, -1).size(1)
    z_dim = mu.size(-1)

    # 采样 k 个 z: shape [B, k, Z]
    eps = torch.randn(B, k, z_dim, device=device)
    std = torch.exp(0.5 * logvar)                          # [B, Z]
    z = mu.unsqueeze(1) + std.unsqueeze(1) * eps           # [B, k, Z]

    # 常量
    dtype = z.dtype
    log_2pi = torch.tensor(math.log(2 * math.pi), device=device, dtype=dtype)
    log_k = torch.tensor(math.log(k), device=device, dtype=dtype)

    # 第二项 log p(z) 标准正态先验: [B, k]
    log_pz = -0.5 * (z**2 + log_2pi).sum(-1)

    # 第三项 log q(z|x): [B, k]
    log_qz = -0.5 * (
        ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
        + logvar.unsqueeze(1)
        + log_2pi
    ).sum(-1)

    # 解码: flatten z -> [B*k, Z] and decode -> [B*k, D]
    z_flat = z.view(B * k, z_dim)
    x_recon = decoder(z_flat)                                # [B*k, D]
    x_expand = x.unsqueeze(1).expand(-1, k, -1).contiguous().view(B * k, -1)
    diff = x_expand - x_recon                                # [B*k, D]

    # 第一项 log p(x|z) 似然，固定方差 sigma2: [B, k]
    const_px = torch.tensor(0.5 * D * math.log(2 * math.pi * sigma2), device=device, dtype=diff.dtype)
    log_px = (-0.5 * (diff**2 / sigma2).sum(-1) - const_px).view(B, k)

    # importance weights: log-sum-exp 聚合 （per data point: [B]）
    log_w = log_px + log_pz - log_qz                        # [B, k]
    m = log_w.max(dim=1, keepdim=True).values               # [B, 1]
    sum_exp = torch.exp(log_w - m).sum(dim=1)               # [B]
    Lk = m.squeeze(1) + torch.log(sum_exp) - log_k          # [B]

    total_loss = - Lk.sum()
    return total_loss


In [19]:
def loss_function(x, recon_x, mu, logvar, prior_var=1.0, sigma2=1.0):
    """
    修改版: 支持动态调整先验方差 prior_var
    prior_var: scalar, 先验分布的方差 (默认1.0)
    """

    # 2. KL 散度 (General Gaussian Prior)
    # log(prior_var)
    log_prior_var = math.log(prior_var)
    
    # 公式推导转代码:
    # 0.5 * sum(log(sigma_p^2) - log(sigma^2) - 1 + (sigma^2 + mu^2)/sigma_p^2)
    # 提取 -0.5: -0.5 * sum(1 + log(sigma^2) - log(sigma_p^2) - (sigma^2 + mu^2)/sigma_p^2)
    
    var = logvar.exp()
    KLD = -0.5 * torch.sum(1 + logvar - log_prior_var - (mu.pow(2) + var) / prior_var)
    
    return KLD

In [20]:
def score_fn_torch(z: torch.Tensor, x_target: torch.Tensor, decoder: nn.Module, sigma2=1.0, prior_var=1.0) -> torch.Tensor:
    """
    修改版: 引入 prior_var 参数
    计算 ∇_z log p(z, x_target)
    """
    assert z.requires_grad, "z must have requires_grad=True"

    B, D = z.shape[0], z.shape[1]
    if x_target.ndim == 1:
        x_target = x_target.unsqueeze(0)
    
    x_target = x_target.view(B, -1)   # [B, 3072]
    x_rec = decoder(z).view(B, -1)
    diff = x_target - x_rec

    # --- 修改点 1: log p(z) 考虑先验方差 ---
    # log p(z) = -0.5 * (z^2 / prior_var)
    logpz = -0.5 * (z ** 2 / prior_var).sum(dim=1)

    # log p(x|z) = -1/(2σ²) * ||x - decoder(z)||²
    logpx = -0.5 / sigma2 * (diff ** 2).sum(dim=1)

    # --- 分别求两项梯度 ---
    # --- 修改点 2: grad_pz 除以先验方差 ---
    grad_pz = -z / prior_var  # [B, D]
    
    grad_px = torch.autograd.grad(logpx.sum(), z, create_graph=True, retain_graph=True)[0]

    # 合并两项得到 g(z)
    g = grad_pz + grad_px  # [B, D]
    
    return g

In [21]:
def bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K=2, lambda_t=0.1, sigma2=1.0, prior_var=1.0, # <--- 新增参数
):

    N, D = mu.shape

    # === 1. 重参数化采样 z (这就是你报错缺失的部分) ===
    std = torch.exp(0.5 * logvar)
    var = std**2
    # 采样 epsilon
    eps = torch.randn(N, sample_K, D, device=mu.device)
    # 得到 z: [N, K, D]
    z = mu.unsqueeze(1) + eps * std.unsqueeze(1)
    # 必须开启梯度，因为后面要算 score
    z.requires_grad_(True)

    # === 2. 计算 score ===
    z_flat = z.view(N * sample_K, D)
    x_rep = x_target.unsqueeze(1)\
                     .expand(N, sample_K, *x_target.shape[1:])\
                     .reshape(N * sample_K, *x_target.shape[1:])
    
    # 调用修改后的 score_fn_torch，传入 prior_var
    scores_flat = score_fn_torch(z_flat, x_rep, decoder, sigma2=sigma2, prior_var=prior_var)
    scores = scores_flat.view(N, sample_K, D)

    # === 计算 Loss 项 ===
    # term1: 因为 prior_var = 1, 所以是 D / 1.0 = D
    
    
    # term2 和 term3 的计算保持原样，它们内部使用的 scores 已经包含了 sigma2 的影响
    g_norm2_sigma = (var.unsqueeze(1) * scores**2).sum(dim=2)
    align_term = ((z - mu.unsqueeze(1)) * scores).sum(dim=2)
    
    term1 = D / prior_var
    term2 = g_norm2_sigma.mean(dim=1)
    term3 = 2 * align_term.mean(dim=1) 

    # 如果你想做敏感性测试，记得加上 term1
    score_div =  term2 + term3
    
    loss = score_div.mean()  # 对 batch 求平均

    return loss

In [22]:
def cv_bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K=2, lambda_t=0.1, sigma2=0.1,
    writer=None, global_step=None,
):
    """
    向量化的 BaM 损失，支持 batch 并行 (集成 Control Variates 版本)。
    """
    N, D = mu.shape
    eps = 1e-8 # 防止除0

    # -------------------------------------------------------------------------
    # [Control Variates 准备阶段] 计算 g(mu) 和 grad(f1(mu))
    # -------------------------------------------------------------------------
    # 确保 mu 计算梯度
    mu.requires_grad_(True)
    
    # 1. 计算 g(mu) [N, D]
    # 注意：这里直接使用 x_target，因为 mu 对应每个 batch 只有 1 个
    g_mu = score_fn_torch(mu, x_target, decoder, sigma2)

    # 2. 计算 f1(mu) = g(mu)^T Sigma g(mu)
    std = torch.exp(0.5 * logvar)
    var = std**2
    f1_mu = (var * g_mu**2).sum(dim=1) # [N]

    # 3. 计算 grad(f1(mu)) [N, D]
    grad_f1_mu = torch.autograd.grad(
        outputs=f1_mu.sum(), inputs=mu, create_graph=True
    )[0]

    # -------------------------------------------------------------------------
    # [原有采样逻辑]
    # -------------------------------------------------------------------------
    # 1. 重参数化采样 z: [N, K, D]
    eps_z = torch.randn(N, sample_K, D, device=mu.device)
    z = mu.unsqueeze(1) + eps_z * std.unsqueeze(1)
    z.requires_grad_(True)

    # 2. 计算 score，先 reshape 到 [N*K, D]
    z_flat = z.view(N * sample_K, D)
    x_rep = x_target.unsqueeze(1)               \
                    .expand(N, sample_K, *x_target.shape[1:]) \
                    .reshape(N * sample_K, *x_target.shape[1:])
    scores_flat = score_fn_torch(z_flat, x_rep, decoder, sigma2)
    scores = scores_flat.view(N, sample_K, D)

    # -------------------------------------------------------------------------
    # [Control Variates 计算阶段]
    # -------------------------------------------------------------------------
    z_minus_mu = z - mu.unsqueeze(1) # [N, K, D]

    # 原始项 (Raw Terms) [N, K]
    # Term 2 raw: ||g_l||_Σ^2
    g_norm2_sigma = (var.unsqueeze(1) * scores**2).sum(dim=2) 
    # Term 3 raw (不含系数2): (z_l - μ)^T * g_l
    align_term = (z_minus_mu * scores).sum(dim=2)

    # 控制变量项 (CV Terms) [N, K]
    # CV1: grad(f1(mu))^T (z - mu)
    cv1_vals = (grad_f1_mu.unsqueeze(1) * z_minus_mu).sum(dim=2)
    # CV2: g(mu)^T (z - mu)
    cv2_vals = (g_mu.unsqueeze(1) * z_minus_mu).sum(dim=2)

    # 计算最优系数 c1, c2 (对 K 维度求统计量) [N]
    # c = Cov(Term, CV) / Var(CV)
    def compute_optimal_c(term_vals, cv_vals):
        # 1. 中心化
        term_centered = term_vals - term_vals.mean(dim=1, keepdim=True)
        cv_centered = cv_vals - cv_vals.mean(dim=1, keepdim=True)
        
        # 2. 分子：Covariance
        numerator = (term_centered * cv_centered).mean(dim=1)
        
        # 3. 分母：Variance
        # 【修改点1】加大 eps，从 1e-8 加到 1e-4 或 1e-5，避免 K=2 时方差过小导致的除零爆炸
        denominator = cv_vals.var(dim=1, unbiased=False) + 1e-5 
        
        c = numerator / denominator
        
        # 【修改点2】Detach！我们只希望 c 作为降低方差的系数，不希望优化器去通过改变 c 来优化 Loss
        c = c.detach()
        
        # 【修改点3】截断 (Clamping)！
        # 如果 c 计算出来是几千几万，说明 CV 和原始项的相关性出了问题或者方差估计完全错误。
        # 限制在 [-100, 100] 或更小的范围内，保证训练初期 Loss 不崩。
        #c = torch.clamp(c, min=-100.0, max=100.0)
        
        return c

    c1 = compute_optimal_c(g_norm2_sigma, cv1_vals)
    c2 = compute_optimal_c(align_term, cv2_vals)

    # -------------------------------------------------------------------------
    # [Score-matching 三项 (应用 CV)]
    # -------------------------------------------------------------------------
    
    # 1) tr(I) = D
    term1 = D
    
    # 2) Term 2: Mean(Raw - c1 * CV)
    # 1/L ∑ ||g_l||_Σ^2 - c1 * 1/L ∑ CV1
    term2_corrected = g_norm2_sigma - c1.unsqueeze(1) * cv1_vals
    term2 = term2_corrected.mean(dim=1) # [N]
    
    # 3) Term 3: Mean(Raw - c2 * CV) * 2
    # 2/L ∑ (z_l - mu)^T g_l - 2 * c2 * 1/L ∑ CV2
    term3_corrected = align_term - c2.unsqueeze(1) * cv2_vals
    term3 = 2 * term3_corrected.mean(dim=1) # [N]

    # 汇总 Loss
    score_div = term2 + term3

    # 6. 最终 loss 对 batch 求平均
    loss = score_div.mean() # 对 batch 求平均

    return loss

### eval

In [23]:
def eval_fid(model, test_loader, device, fid_metric):
    model.eval()
    fid_metric.reset()

    with torch.no_grad():
        for real, _ in test_loader:
            real = real.to(device)

            # 编码—采样—解码
            mu, logvar = model.encode(real)
            z = model.reparameterize(mu, logvar)
            fake = model.decode(z)

            # 从 [B, D] 恢复成 [B,3,side,side]
            B, D = real.shape
            side = int(D**0.5)
            real_img = real.view(B, 1, side, side).repeat(1, 3, 1, 1)
            fake_img = fake.view(B, 1, side, side).repeat(1, 3, 1, 1)

            # resize 到 299×299
            real_rs = F.interpolate(
                real_img, size=(299, 299), mode="bilinear", align_corners=False
            )
            fake_rs = F.interpolate(
                fake_img, size=(299, 299), mode="bilinear", align_corners=False
            )

            # 更新 FID 统计（uint8）
            fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
            fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)

    # 计算并返回得分
    fid_score = fid_metric.compute().item()
    model.train()
    return fid_score

In [24]:
# def eval_fid(model, test_loader, device, fid_metric):
#     model.eval()
#     fid_metric.reset()

#     with torch.no_grad():
#         for real, _ in test_loader:
#             real = real.to(device)

#             # 编码—采样—解码
#             mu, logvar = model.encode(real)
#             z = model.reparameterize(mu, logvar)
#             fake = model.decode(z)

#             # 从 [B, D] 恢复成 [B,3,side,side]
#             B, D = real.shape
#             #side = int(D**0.5)
#             real_img = real.view(B, 3, s, s)   # 如果 flatten 过，需要 reshape 回来
#             fake_img = fake.view(B, 3, s, s)


#             # resize 到 299×299
#             real_rs = nn.functional.interpolate(real_img, size=(299, 299), mode="bilinear", align_corners=False)
#             fake_rs = nn.functional.interpolate(fake_img, size=(299, 299), mode="bilinear", align_corners=False)


#             # 更新 FID 统计（uint8）
#             fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
#             fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)

#     # 计算并返回得分
#     fid_score = fid_metric.compute().item()
#     model.train()
#     return fid_score

In [25]:
import math
@torch.no_grad()
def evaluate_iwae_gauss(model, dataloader, device, k=5000, sigma2=1.0):
    total_loglik = 0.0
    total_samples = 0
    D = None

    for x, _ in dataloader:
        x = x.to(device)
        B, D = x.shape
        total_samples += B

        # 编码
        mu, logvar = model.encode(x)  # [B, Z]
        std = torch.exp(0.5 * logvar)  # [B, Z]

        # 采样 k 个 z
        eps = torch.randn(B, k, mu.size(1), device=device)
        z = mu.unsqueeze(1) + std.unsqueeze(1) * eps  # [B, k, Z]

        # 第二项：log p(z) 标准正态先验
        log_pz = -0.5 * (z**2 + math.log(2 * math.pi)).sum(-1)  # [B, k]

        # 第三项：log q(z|x)
        log_qz = -0.5 * (
            ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
            + logvar.unsqueeze(1)
            + math.log(2 * math.pi)
        ).sum(
            -1
        )  # [B, k]

        # 解码
        x_recon = model.decode(z.view(B * k, -1))  # [B*k, D]
        x_expand = x.unsqueeze(1).repeat(1, k, 1).view(B * k, -1)
        diff = x_expand - x_recon  # [B*k, D]

        # 第一项：log p(x|z) 似然，固定方差 sigma2
        log_px = (
            -0.5 * (diff**2 / sigma2).sum(-1) - 0.5 * D * math.log(2 * math.pi * sigma2)
        ).view(
            B, k
        )  # [B, k]

        # importance weights: log-sum-exp 聚合
        log_w = log_px + log_pz - log_qz  # [B, k]
        m = log_w.max(dim=1, keepdim=True).values  # [B, 1]
        Lk = (
            m.squeeze(1) + torch.log(torch.exp(log_w - m).sum(dim=1)) - math.log(k)
        )  # [B]

        total_loglik += Lk.sum().item()

    avg_Lk = total_loglik / total_samples
    avg_NLL = -avg_Lk
    bpd = avg_NLL / (D * math.log(2))  # bits/dim
    return avg_Lk, avg_NLL, bpd

### plt

In [26]:
# gary
def write_test_images(writer, model, data_loader, device, global_step, sample_K=7,folder_name="img"):
    """
    将测试输入和重构结果保存到 TensorBoard。

    参数：
    - writer: SummaryWriter 实例
    - model: VAE 模型
    - data_loader: DataLoader，用于获取一个 batch
    - device: 运行设备（cuda or cpu）
    - global_step: TensorBoard 全局步数
    - sample_K: 最多展示的样本数
    """
    model.eval()
    batch, _ = next(iter(data_loader))
    batch = batch.to(device)
    K = min(sample_K, batch.size(0))
    inputs = batch[:K].cpu().view(K, 28, 28)

    # 编码 -> 采样 -> 解码
    with torch.no_grad():
        mu, logvar = model.encode(batch)
        sigma = torch.exp(0.5 * logvar)
        eps = torch.randn_like(sigma)

        z = mu + eps * sigma
        recon = model.decode(z[:K]).cpu().view(K, 28, 28).numpy()

    # 合并原图和重构图到一张图
    fig = plt.figure(figsize=(15, 6))
    for i in range(K):
        # 第一行：原图
        ax = fig.add_subplot(2, K, i + 1)
        ax.imshow(inputs[i], cmap="gray")
        ax.axis("off")
        # 第二行：重构图
        ax = fig.add_subplot(2, K, K + i + 1)
        ax.imshow(recon[i], cmap="gray")
        ax.axis("off")

    writer.add_figure(f"{folder_name}/{global_step}-rec", fig, global_step)
    plt.close(fig)

In [27]:
# # RGB
# def write_test_images(
#     writer,
#     model,
#     data_loader,
#     device,
#     global_step,
#     sample_K=7,
#     img_size=(s, s),
#     num_channels=3,
#     folder_name="img",
# ):
#     """
#     在 TensorBoard 上展示 CIFAR-10 原图 & VAE 重构图。

#     参数
#     - writer: SummaryWriter
#     - model: VAE（输入维度 3072）
#     - data_loader: DataLoader，产出 (flat_img, label)，flat_img shape [B,3072]
#     - device: torch.device
#     - global_step: 整数步数
#     - sample_K: 最多展示多少张
#     - img_size: (H, W)，这里 (32,32)
#     - num_channels: 通道数，这里 3
#     """
#     model.eval()

#     # 1) 从 data_loader 累积扁平张量 flat_imgs 直到 >= sample_K
#     all_flat = []
#     for flat_img, _ in data_loader:
#         all_flat.append(flat_img)
#         if sum(x.size(0) for x in all_flat) >= sample_K:
#             break
#     flat_imgs = torch.cat(all_flat, dim=0).to(device)  # 至少 sample_K 张

#     B = flat_imgs.size(0)
#     K = min(sample_K, B)
#     C, H, W = num_channels, *img_size

#     # 2) 把前 K 张先恢复成 [K, C, H, W] 供可视化
#     inputs = flat_imgs[:K].view(K, C, H, W).cpu()

#     # 3) 编码 → 样本 → 解码（全程基于扁平向量）
#     with torch.no_grad():
#         mu, logvar = model.encode(flat_imgs)  # [B, latent_dim]×2
#         sigma = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(sigma)
#         z = mu + eps * sigma

#         recon_flat = model.decode(z[:K]).cpu()  # [K, 3072]
#         recon = recon_flat.view(K, C, H, W)  # 还原成 [K, C, H, W]

#     # 4) 转成 HWC NumPy 数组
#     inputs_np = inputs.permute(0, 2, 3, 1).contiguous().numpy()  # [K, H, W, C]
#     recon_np = recon.permute(0, 2, 3, 1).contiguous().numpy()  # [K, H, W, C]

#     # 5) 按组绘图，每组最多 7 张
#     group_size = 7
#     for group_idx in range(0, K, group_size):
#         end_idx = min(group_idx + group_size, K)
#         current_K = end_idx - group_idx

#         fig = plt.figure(figsize=(2 * current_K, 4))
#         for i in range(current_K):
#             # 上排：原图
#             ax1 = fig.add_subplot(2, current_K, i + 1)
#             ax1.imshow(inputs_np[group_idx + i])
#             ax1.axis("off")

#             # 下排：重构图
#             ax2 = fig.add_subplot(2, current_K, current_K + i + 1)
#             ax2.imshow(recon_np[group_idx + i])
#             ax2.axis("off")

#         writer.add_figure(
#             f"{folder_name}/{global_step}-{group_idx // group_size}",
#             fig,
#             global_step,
#         )
#         plt.close(fig)
#     model.train()

In [28]:
def log_latent_space(writer, model, dataloader, global_step, device):
    model.eval()
    all_mu = []
    all_logvar = []
    all_labels = []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            mu, logvar = model.encode(x)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_labels.extend(y.cpu().tolist())

    all_mu = torch.cat(all_mu, dim=0)  # [N, z_dim]
    all_logvar = torch.cat(all_logvar, dim=0)  # [N, z_dim]
    all_labels = [str(l) for l in all_labels]  # 转成字符串，方便 Color by

    max_points = 2000
    all_mu = all_mu[:max_points]
    all_labels = all_labels[:max_points]
    all_logvar = all_logvar[:max_points]

    # print("mu 是否有nan: ", torch.isnan(all_mu).any(), torch.isinf(all_mu).any())
    # print("logvar 是否有nan: ", torch.isnan(all_logvar).any(), torch.isinf(all_logvar).any())
    # print("数据维度: ", all_mu.shape, all_logvar.shape, len(all_labels))
    # print("数据范围: ", all_mu.min().item(), all_mu.max().item(), all_logvar.min().item(), all_logvar.max().item())
    # print("数据方差: ", all_mu.std(dim=0), all_logvar.std(dim=0))

    # print("标签类别: ", set(all_labels))
    # 写入 TensorBoard Embedding
    writer.add_embedding(
        mat=all_mu,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_mu",
        global_step=global_step,
    )
    writer.add_embedding(
        mat=all_logvar,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_logvar",
        global_step=global_step,
    )

In [29]:
def log_latent_stats(model, loader, writer, epoch, device):
    model.eval()
    ############## 写 mu/std/z m##############
    all_mu, all_std, all_z = [], [], []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            mu, logvar = model.encode(x)  # [B, Z]
            std = torch.exp(0.5 * logvar)  # [B, Z]
            # 如果你想按重参数化采样的 z
            eps = torch.randn_like(std)
            z = mu + eps * std  # [B, Z]

            all_mu.append(mu.cpu())
            all_std.append(std.cpu())
            all_z.append(z.cpu())

    # 拼接成 [N, Z]
    all_mu = torch.cat(all_mu, dim=0)
    all_std = torch.cat(all_std, dim=0)
    all_z = torch.cat(all_z, dim=0)

    # 对样本维度取均值 → [Z]
    mu_mean = all_mu.mean(dim=0)  # Tensor of shape [Z]
    std_mean = all_std.mean(dim=0)
    z_mean = all_z.mean(dim=0)

    # 控制台打印（可选）
    print(f"Epoch {epoch+1}  mu_mean per dim:  {mu_mean.tolist()}")
    print(f"Epoch {epoch+1}  std_mean per dim: {std_mean.tolist()}")
    print(f"Epoch {epoch+1}  z_mean per dim:   {z_mean.tolist()}")

    # 写入 TensorBoard：每个潜变量维度一个 scalar
    for i in range(mu_mean.size(0)):
        writer.add_scalar(f"latent/mu_dim{i}", mu_mean[i].item(), epoch)
        writer.add_scalar(f"latent/std_dim{i}", std_mean[i].item(), epoch)
        writer.add_scalar(f"latent/z_dim{i}", z_mean[i].item(), epoch)

    # 如果还想看分布，可写 histogram：
    writer.add_histogram("histogram/mu", all_mu, epoch)
    writer.add_histogram("histogram/std", all_std, epoch)
    writer.add_histogram("histogram/z", all_z, epoch)

    ############## 写 g_dim ##############
    Z = all_mu.size(1)  # 潜变量维度 5

    for d in range(Z):
        vals = epoch_metrics_g[f"g_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均梯度
        writer.add_scalar(f"latent_g/g_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g/g_dim{d}", avg, epoch)

    # 写 g 的整体分布直方图
    all_g = []
    for d in range(Z):
        all_g.extend(epoch_metrics_g[f"g_dim{d}"])  # 收集所有维度的 g

    all_g_tensor = torch.tensor(all_g, dtype=torch.float32)
    writer.add_histogram("histogram_g/g", all_g_tensor, epoch)

    print(f"Epoch {epoch+1}  g per dim: {[epoch_metrics_g[f'g_abs_dim{d}'][-1] for d in range(Z)]}")

    ############### 写 g 的分解项 ##############
    for d in range(Z):
        vals = epoch_metrics_g[f"g_pz_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均
        writer.add_scalar(f"latent_g_pz/g_pz_abs_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_abs_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_pz_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_pz/g_pz_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_dim{d}", avg, epoch)

    # g_pz 和 g_px 的整体分布直方图
    all_g_pz = []
    all_g_px = []
    for d in range(Z):            
        all_g_pz.extend(epoch_metrics_g[f"g_pz_dim{d}"])
        all_g_px.extend(epoch_metrics_g[f"g_px_dim{d}"])

    all_g_pz_tensor = torch.tensor(all_g_pz, dtype=torch.float32)
    all_g_px_tensor = torch.tensor(all_g_px, dtype=torch.float32)
    writer.add_histogram("histogram_g/g_pz", all_g_pz_tensor, epoch)
    writer.add_histogram("histogram_g/g_px", all_g_px_tensor, epoch)

    print(f"Epoch {epoch+1}  g_pz per dim: {[epoch_metrics_g[f'g_pz_abs_dim{d}'][-1] for d in range(Z)]}")
    print(f"Epoch {epoch+1}  g_px per dim: {[epoch_metrics_g[f'g_px_abs_dim{d}'][-1] for d in range(Z)]}")

    model.train()  # 别忘了恢复训练模式

## run mnist


In [32]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
prior_var0 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1
lambda_D0 = 0.5
lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = False
flag_iwae = False
flag_cv = False
prefix = f"F3-add-latent{LATENT_DIM}"
folder = "runs__tsne"
file_idx = 4444
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1.0 # BaM 的 KL{t+1} 的分子

sigma20 = 1
prior_var0 = 1
sigma2_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
prior_var_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

for lr0 in [1e-3]:
    for sigma20 in [1]:
        for prior_var0 in prior_var_list:    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{num_KLt/lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar, sigma2=sigma20, prior_var=prior_var0)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    prior_var=prior_var0)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = (1.0 / sigma20) * MSE

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= num_KLt / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        elif flag_vae:
                            loss = MSE + KL
                        else:
                            loss = MSE + sum_Dq_plus_lKLt_dot_KLt


                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    # mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    # writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {'N/A'}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                

Using device: cuda:2
[Epoch 1] Loss: 195.8779, FID: 280.5645, BCE: N/A
           MSE: 72.9163, Dq: 227.2456, KL: 7.8521, KL_t: 466.9415
[Epoch 2] Loss: 45.1540, FID: 250.4357, BCE: N/A
           MSE: 44.2792, Dq: 1.3238, KL: 6.6269, KL_t: 10.6423
[Epoch 3] Loss: 37.0566, FID: 228.4879, BCE: N/A
           MSE: 36.6421, Dq: 0.4964, KL: 12.8369, KL_t: 8.3152
[Epoch 4] Loss: 34.6570, FID: 214.3001, BCE: N/A
           MSE: 34.5702, Dq: 0.0013, KL: 15.3091, KL_t: 4.3064
[Epoch 5] Loss: 33.8223, FID: 207.7145, BCE: N/A
           MSE: 33.7781, Dq: -0.1021, KL: 17.1243, KL_t: 4.7610
[Epoch 6] Loss: 33.3906, FID: 200.2200, BCE: N/A
           MSE: 33.2862, Dq: -0.0886, KL: 18.1687, KL_t: 7.4343
[Epoch 7] Loss: 32.6769, FID: 195.8041, BCE: N/A
           MSE: 32.5921, Dq: -0.0790, KL: 19.5494, KL_t: 6.2141
[Epoch 8] Loss: 32.0762, FID: 176.9067, BCE: N/A
           MSE: 31.9092, Dq: -0.0389, KL: 20.7619, KL_t: 9.3250
[Epoch 9] Loss: 31.1695, FID: 169.1307, BCE: N/A
           MSE: 31.0217, D

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-2o2tne1y'


[Epoch 10] Loss: 30.3298, FID: 153.1938, BCE: N/A
           MSE: 30.1484, Dq: 0.0003, KL: 22.6846, KL_t: 9.0633
[Epoch 11] Loss: 29.4697, FID: 145.7995, BCE: N/A
           MSE: 29.2863, Dq: 0.0116, KL: 23.8539, KL_t: 8.8792
[Epoch 12] Loss: 28.7356, FID: 133.8459, BCE: N/A
           MSE: 28.5358, Dq: 0.0196, KL: 24.6292, KL_t: 9.4985
[Epoch 13] Loss: 28.0056, FID: 120.6863, BCE: N/A
           MSE: 27.8023, Dq: 0.0337, KL: 25.7099, KL_t: 9.3245
[Epoch 14] Loss: 27.4212, FID: 113.9957, BCE: N/A
           MSE: 27.1905, Dq: 0.0570, KL: 25.9897, KL_t: 10.1098
[Epoch 15] Loss: 26.7629, FID: 103.1650, BCE: N/A
           MSE: 26.5527, Dq: 0.0570, KL: 27.4934, KL_t: 9.0831
[Epoch 16] Loss: 26.2843, FID: 95.2271, BCE: N/A
           MSE: 26.0472, Dq: 0.0821, KL: 28.6799, KL_t: 9.8007
[Epoch 17] Loss: 25.8279, FID: 93.3575, BCE: N/A
           MSE: 25.5872, Dq: 0.0885, KL: 29.2256, KL_t: 9.8185
[Epoch 18] Loss: 25.3819, FID: 87.3164, BCE: N/A
           MSE: 25.1241, Dq: 0.1118, KL: 30.3938

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-vrd40vft'


[Epoch 27] Loss: 22.6185, FID: 66.9914, BCE: N/A
           MSE: 22.3323, Dq: 0.1555, KL: 36.0039, KL_t: 10.4253
[Epoch 28] Loss: 22.4399, FID: 67.1381, BCE: N/A
           MSE: 22.1445, Dq: 0.1617, KL: 36.5610, KL_t: 10.7279
[Epoch 29] Loss: 22.2283, FID: 66.2647, BCE: N/A
           MSE: 21.9489, Dq: 0.1495, KL: 36.5305, KL_t: 10.2299
[Epoch 30] Loss: 22.0492, FID: 63.7691, BCE: N/A
           MSE: 21.7724, Dq: 0.1465, KL: 37.1589, KL_t: 10.1794
[Epoch 31] Loss: 21.9247, FID: 63.0529, BCE: N/A
           MSE: 21.6522, Dq: 0.1404, KL: 36.9061, KL_t: 10.1130
[Epoch 32] Loss: 21.7831, FID: 62.8940, BCE: N/A
           MSE: 21.4792, Dq: 0.1676, KL: 37.8872, KL_t: 11.0094
[Epoch 33] Loss: 21.6327, FID: 64.1680, BCE: N/A
           MSE: 21.3492, Dq: 0.1462, KL: 37.9069, KL_t: 10.5200
[Epoch 34] Loss: 21.4617, FID: 61.0292, BCE: N/A
           MSE: 21.1896, Dq: 0.1377, KL: 38.5881, KL_t: 10.1613
[Epoch 35] Loss: 21.3435, FID: 60.3568, BCE: N/A
           MSE: 21.0648, Dq: 0.1392, KL: 38.862

...

[Epoch 52] Loss: 16.3188, FID: 38.1920, BCE: N/A
           MSE: 16.4986, Dq: -0.4731, KL: 19.1405, KL_t: 2.8363
[Epoch 53] Loss: 16.2787, FID: 38.2982, BCE: N/A
           MSE: 16.4524, Dq: -0.4578, KL: 19.2763, KL_t: 2.7611
[Epoch 54] Loss: 16.2435, FID: 38.0612, BCE: N/A
           MSE: 16.4089, Dq: -0.4410, KL: 19.3469, KL_t: 2.7516
[Epoch 55] Loss: 16.2278, FID: 37.4037, BCE: N/A
           MSE: 16.4121, Dq: -0.4804, KL: 19.1570, KL_t: 2.7922
[Epoch 56] Loss: 16.2094, FID: 37.6133, BCE: N/A
           MSE: 16.3841, Dq: -0.4604, KL: 19.2569, KL_t: 2.7721
[Epoch 57] Loss: 16.2008, FID: 38.4214, BCE: N/A
           MSE: 16.3766, Dq: -0.4659, KL: 19.2188, KL_t: 2.8593
[Epoch 58] Loss: 16.1625, FID: 37.7788, BCE: N/A
           MSE: 16.3361, Dq: -0.4597, KL: 19.2608, KL_t: 2.8125
[Epoch 59] Loss: 16.1292, FID: 36.7761, BCE: N/A
           MSE: 16.3091, Dq: -0.4681, KL: 19.2455, KL_t: 2.7101
[Epoch 60] Loss: 16.1281, FID: 36.8291, BCE: N/A
           MSE: 16.3045, Dq: -0.4638, KL: 19.26

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-_66ip7_6'


[Epoch 63] Loss: 16.0538, FID: 36.8798, BCE: N/A
           MSE: 16.2230, Dq: -0.4545, KL: 19.3194, KL_t: 2.9021
[Epoch 64] Loss: 16.0378, FID: 37.5913, BCE: N/A
           MSE: 16.2239, Dq: -0.4827, KL: 19.1622, KL_t: 2.7642
[Epoch 65] Loss: 16.0107, FID: 36.3233, BCE: N/A
           MSE: 16.1874, Dq: -0.4655, KL: 19.3182, KL_t: 2.8030
[Epoch 66] Loss: 15.9991, FID: 37.1593, BCE: N/A
           MSE: 16.1929, Dq: -0.4990, KL: 19.1825, KL_t: 2.7821
[Epoch 67] Loss: 15.9920, FID: 36.7166, BCE: N/A
           MSE: 16.1760, Dq: -0.4804, KL: 19.3068, KL_t: 2.8129
[Epoch 68] Loss: 15.9751, FID: 37.4589, BCE: N/A
           MSE: 16.1661, Dq: -0.4966, KL: 19.2367, KL_t: 2.8649
[Epoch 69] Loss: 15.9482, FID: 35.8323, BCE: N/A
           MSE: 16.1271, Dq: -0.4725, KL: 19.4183, KL_t: 2.8670
[Epoch 70] Loss: 15.9266, FID: 36.0051, BCE: N/A
           MSE: 16.1057, Dq: -0.4718, KL: 19.4174, KL_t: 2.8360
[Epoch 71] Loss: 15.9260, FID: 36.0146, BCE: N/A
           MSE: 16.1129, Dq: -0.4883, KL: 19.36

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-upzdg9w0'


[Epoch 93] Loss: 15.6087, FID: 34.4009, BCE: N/A
           MSE: 15.7943, Dq: -0.4836, KL: 19.7791, KL_t: 2.8080
[Epoch 94] Loss: 15.5978, FID: 34.6968, BCE: N/A
           MSE: 15.7857, Dq: -0.4910, KL: 19.7693, KL_t: 2.8797
[Epoch 95] Loss: 15.5797, FID: 34.8662, BCE: N/A
           MSE: 15.7615, Dq: -0.4775, KL: 19.8203, KL_t: 2.8500
[Epoch 96] Loss: 15.5637, FID: 34.7824, BCE: N/A
           MSE: 15.7461, Dq: -0.4767, KL: 19.8777, KL_t: 2.7991
[Epoch 97] Loss: 15.5749, FID: 34.6691, BCE: N/A
           MSE: 15.7698, Dq: -0.5071, KL: 19.6506, KL_t: 2.9303
[Epoch 98] Loss: 15.5372, FID: 34.5453, BCE: N/A
           MSE: 15.7298, Dq: -0.4976, KL: 19.7747, KL_t: 2.8158
[Epoch 99] Loss: 15.5338, FID: 34.7627, BCE: N/A
           MSE: 15.7208, Dq: -0.4850, KL: 19.8595, KL_t: 2.7765
[Epoch 100] Loss: 15.5183, FID: 33.6111, BCE: N/A
           MSE: 15.7023, Dq: -0.4791, KL: 19.9603, KL_t: 2.7745
Final Test FID: 32.9145


In [33]:
#### vae
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
prior_var0 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1
lambda_D0 = 0.5
lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = True
flag_iwae = False
flag_cv = False
prefix = f"F3-add-latent{LATENT_DIM}"
folder = "runs__tsne"
file_idx = 4444
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1.0 # BaM 的 KL{t+1} 的分子

sigma20 = 1
prior_var0 = 1
sigma2_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
prior_var_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

for lr0 in [1e-3]:
    for sigma20 in [1]:
        for prior_var0 in prior_var_list:    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{num_KLt/lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar, sigma2=sigma20, prior_var=prior_var0)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    prior_var=prior_var0)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = (1.0 / sigma20) * MSE

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= num_KLt / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        elif flag_vae:
                            loss = MSE + KL
                        else:
                            loss = MSE + sum_Dq_plus_lKLt_dot_KLt


                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    # mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    # writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {'N/A'}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                

Using device: cuda:2
[Epoch 1] Loss: 52.3622, FID: 239.2445, BCE: N/A
           MSE: 47.0631, Dq: 0.0000, KL: 5.2991, KL_t: 775.5289
[Epoch 2] Loss: 42.8861, FID: 228.5506, BCE: N/A
           MSE: 37.4214, Dq: 0.0000, KL: 5.4648, KL_t: 3.1963


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-3gvlf3vc'


[Epoch 3] Loss: 42.1576, FID: 207.5390, BCE: N/A
           MSE: 36.6824, Dq: 0.0000, KL: 5.4752, KL_t: 1.5938
[Epoch 4] Loss: 41.9767, FID: 205.7293, BCE: N/A
           MSE: 36.4532, Dq: 0.0000, KL: 5.5235, KL_t: 1.3711
[Epoch 5] Loss: 41.8239, FID: 200.6647, BCE: N/A
           MSE: 36.2903, Dq: 0.0000, KL: 5.5336, KL_t: 1.9613
[Epoch 6] Loss: 41.6775, FID: 205.9480, BCE: N/A
           MSE: 36.1437, Dq: 0.0000, KL: 5.5338, KL_t: 2.2042
[Epoch 7] Loss: 41.7129, FID: 197.8932, BCE: N/A
           MSE: 36.1406, Dq: 0.0000, KL: 5.5723, KL_t: 1.6635
[Epoch 8] Loss: 41.5404, FID: 183.8724, BCE: N/A
           MSE: 35.9572, Dq: 0.0000, KL: 5.5832, KL_t: 2.7023
[Epoch 9] Loss: 41.5992, FID: 189.4901, BCE: N/A
           MSE: 35.9842, Dq: 0.0000, KL: 5.6150, KL_t: 1.6667
[Epoch 10] Loss: 41.4629, FID: 170.7798, BCE: N/A
           MSE: 35.8332, Dq: 0.0000, KL: 5.6297, KL_t: 1.7295
[Epoch 11] Loss: 41.3987, FID: 180.0856, BCE: N/A
           MSE: 35.7318, Dq: 0.0000, KL: 5.6669, KL_t: 1.8201

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-o1o0p9z8'


[Epoch 58] Loss: 34.3945, FID: 90.4716, BCE: N/A
           MSE: 26.7034, Dq: 0.0000, KL: 7.6910, KL_t: 1.8453
[Epoch 59] Loss: 34.3245, FID: 87.3154, BCE: N/A
           MSE: 26.6248, Dq: 0.0000, KL: 7.6997, KL_t: 1.6465
[Epoch 60] Loss: 34.3502, FID: 84.7166, BCE: N/A
           MSE: 26.6458, Dq: 0.0000, KL: 7.7044, KL_t: 1.4087
[Epoch 61] Loss: 34.2696, FID: 85.1330, BCE: N/A
           MSE: 26.5402, Dq: 0.0000, KL: 7.7294, KL_t: 2.0952
[Epoch 62] Loss: 34.2189, FID: 88.6852, BCE: N/A
           MSE: 26.4943, Dq: 0.0000, KL: 7.7246, KL_t: 1.7915
[Epoch 63] Loss: 34.1472, FID: 88.8946, BCE: N/A
           MSE: 26.4199, Dq: 0.0000, KL: 7.7274, KL_t: 2.5005
[Epoch 64] Loss: 34.1566, FID: 91.9687, BCE: N/A
           MSE: 26.4035, Dq: 0.0000, KL: 7.7531, KL_t: 2.3265


...

[Epoch 57] Loss: 29.0568, FID: 48.5837, BCE: N/A
           MSE: 20.0903, Dq: 0.0000, KL: 8.9665, KL_t: 0.6189
[Epoch 58] Loss: 29.0416, FID: 50.2089, BCE: N/A
           MSE: 20.0654, Dq: 0.0000, KL: 8.9762, KL_t: 0.7064
[Epoch 59] Loss: 29.0134, FID: 49.7290, BCE: N/A
           MSE: 20.0315, Dq: 0.0000, KL: 8.9819, KL_t: 0.6490
[Epoch 60] Loss: 29.0004, FID: 48.2219, BCE: N/A
           MSE: 20.0011, Dq: 0.0000, KL: 8.9994, KL_t: 0.6361
[Epoch 61] Loss: 29.0008, FID: 48.6343, BCE: N/A
           MSE: 20.0061, Dq: 0.0000, KL: 8.9947, KL_t: 0.6473
[Epoch 62] Loss: 28.9996, FID: 48.9249, BCE: N/A
           MSE: 19.9887, Dq: 0.0000, KL: 9.0109, KL_t: 0.6565
[Epoch 63] Loss: 28.9516, FID: 48.7414, BCE: N/A
           MSE: 19.9420, Dq: 0.0000, KL: 9.0096, KL_t: 0.6367
[Epoch 64] Loss: 28.9583, FID: 48.4272, BCE: N/A
           MSE: 19.9327, Dq: 0.0000, KL: 9.0256, KL_t: 0.6435
[Epoch 65] Loss: 28.9385, FID: 48.4219, BCE: N/A
           MSE: 19.9252, Dq: 0.0000, KL: 9.0134, KL_t: 0.7156
[

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-sk7lm7cg'


[Epoch 83] Loss: 28.7216, FID: 46.9790, BCE: N/A
           MSE: 19.6190, Dq: 0.0000, KL: 9.1026, KL_t: 0.6753
[Epoch 84] Loss: 28.7021, FID: 47.0623, BCE: N/A
           MSE: 19.5933, Dq: 0.0000, KL: 9.1087, KL_t: 0.5715
[Epoch 85] Loss: 28.6906, FID: 47.3836, BCE: N/A
           MSE: 19.5779, Dq: 0.0000, KL: 9.1127, KL_t: 0.7174
[Epoch 86] Loss: 28.7120, FID: 46.9259, BCE: N/A
           MSE: 19.5630, Dq: 0.0000, KL: 9.1490, KL_t: 0.5941
[Epoch 87] Loss: 28.6695, FID: 47.2438, BCE: N/A
           MSE: 19.5375, Dq: 0.0000, KL: 9.1321, KL_t: 0.6043
[Epoch 88] Loss: 28.6600, FID: 47.1403, BCE: N/A
           MSE: 19.5326, Dq: 0.0000, KL: 9.1274, KL_t: 0.6038
[Epoch 89] Loss: 28.6775, FID: 46.4892, BCE: N/A
           MSE: 19.5372, Dq: 0.0000, KL: 9.1403, KL_t: 0.6318
[Epoch 90] Loss: 28.6667, FID: 47.0737, BCE: N/A
           MSE: 19.5187, Dq: 0.0000, KL: 9.1480, KL_t: 0.6952
[Epoch 91] Loss: 28.6409, FID: 46.8963, BCE: N/A
           MSE: 19.4893, Dq: 0.0000, KL: 9.1515, KL_t: 0.6633
[

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-_35k49h4'


[Epoch 22] Loss: 30.1168, FID: 56.2171, BCE: N/A
           MSE: 21.4662, Dq: 0.0000, KL: 8.6506, KL_t: 0.6481
[Epoch 23] Loss: 30.0505, FID: 55.4171, BCE: N/A
           MSE: 21.3908, Dq: 0.0000, KL: 8.6597, KL_t: 0.5798
[Epoch 24] Loss: 29.9788, FID: 57.0267, BCE: N/A
           MSE: 21.3086, Dq: 0.0000, KL: 8.6702, KL_t: 0.6107
[Epoch 25] Loss: 29.9211, FID: 54.5195, BCE: N/A
           MSE: 21.2281, Dq: 0.0000, KL: 8.6930, KL_t: 0.6142
[Epoch 26] Loss: 29.8619, FID: 54.0486, BCE: N/A
           MSE: 21.1501, Dq: 0.0000, KL: 8.7117, KL_t: 0.5771
[Epoch 27] Loss: 29.8200, FID: 54.2713, BCE: N/A
           MSE: 21.0991, Dq: 0.0000, KL: 8.7209, KL_t: 0.5835
[Epoch 28] Loss: 29.7761, FID: 53.2078, BCE: N/A
           MSE: 21.0294, Dq: 0.0000, KL: 8.7467, KL_t: 0.5656
[Epoch 29] Loss: 29.7445, FID: 53.2801, BCE: N/A
           MSE: 20.9681, Dq: 0.0000, KL: 8.7764, KL_t: 0.6405
[Epoch 30] Loss: 29.6733, FID: 53.6423, BCE: N/A
           MSE: 20.9060, Dq: 0.0000, KL: 8.7673, KL_t: 0.4996
[

## eof